In [11]:
import torch

In [13]:
# 基本示例: 手动计算 vs 自动求导
x = torch.tensor(2.0,requires_grad=True)
y =  x ** 2 + 3 * x + 1
print(f"x:{x}\n")
print(f"y = x^2 + 3x + 1 = {y}\n")

# 反向传播,自动求导
# 手动计算导数: y = 2x + 3
y.backward()
print(f"自动求导的结果:{x.grad}\n")
print(f"手动计算导数结果:{2 * x + 3}")


x:2.0

y = x^2 + 3x + 1 = 11.0

自动求导的结果:7.0

手动计算导数结果:7.0


In [21]:
# 1.创建张量是指定 requires_grad=True
from this import d


x1 = torch.tensor([1.0,2.0,3.0],requires_grad=True)
x2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=False)
print(f"x1:{x1.requires_grad}\n")
print(f"x2:{x2.requires_grad}\n")

x3 = torch.tensor([1.0,2.0,3.0])
print(f"修改之前的x3.requires_grad:{x3.requires_grad}\n")
x3.requires_grad = True
print(f"修改之后的x3.requires_grad:{x3.requires_grad}\n")

# y1 = x1+x2+x3 
y1 =  x1.sum()
y2 = x3.sum()

# y1 求导 = 1 + 1 + 1 = 3
y1.backward()
y2.backward()

print(f"y1的梯度:{x1.grad}\n")
print(f"y2的梯度:{x3.grad}\n")

# 混合计算示例
a = torch.tensor(2.0,requires_grad=True)
b = torch.tensor(3.0, requires_grad=False)
c = a * b
print(f"c.requires_grad:{c.requires_grad}\n")

d = torch.tensor(4.0,requires_grad=False)
e = b * d
print(f"e.requires_grad:{e.requires_grad}\n")



x1:True

x2:False

修改之前的x3.requires_grad:False

修改之后的x3.requires_grad:True

y1的梯度:tensor([1., 1., 1.])

y2的梯度:tensor([1., 1., 1.])

c.requires_grad:True

e.requires_grad:False



In [ ]:
# 计算图与梯度累积
# 1,基本计算图
x = torch.tensor(2.0,requires_grad=True)
y = x ** 2
z = y * 3
w = z + 1

# 反向传播计算梯度
w.backward()
print(f"x的梯度:{x.grad}\n") # dw/dx = d(3x^2+1)/dx = 6x

x的梯度:12.0



In [ ]:
# 梯度累计 (训练模型的时候用于优化. gradient accumlation)
x = torch.tensor(2.0,requires_grad=True)

# 第一次计算 dy1/dx = 2x = 4
y1 =  x ** 2 # 这里可以看作正向传播
y1.backward() # 反向传播求梯度
print(f"第一次反向传播之后对应的梯度:{x.grad}\n") 

# 第二次计算 dy2/dx = 2x + 1 = 7
y2 = x * 3 # 又是一次正向传播
y2.backward() # 反向传播求梯度,梯度累计
print(f"第二次反向传播之后对应的梯度:{x.grad}\n")

# 清除梯度
x.grad.zero_()
print(f"清除梯度之后的梯度:{x.grad}\n")

y3 = x ** 2
y3.backward() # 反向传播求梯度
print(f"重新计算梯度之后的结果:{x.grad}\n")


第一次反向传播之后对应的梯度:4.0

第二次反向传播之后对应的梯度:7.0

清除梯度之后的梯度:0.0

第三次反向传播之后对应的梯度:4.0



In [26]:
# backward() 方法
# 1，标量输出的反向传播(最常见的请求),m1 dl loos 往往求平均或加合得到的

x = torch.tensor([1,0,2.0,3.0],requires_grad=True)
y = x.sum()
y.backward()
print(f"标量y输出的梯度:{x.grad}\n")


标量y输出的梯度:tensor([1., 1., 1., 1.])



In [31]:
# 2，向量输出的反向传播 (backward 需要参数)
x = torch.tensor([1.0,2.0,3.0],requires_grad=True)
y = x ** 2
print(f"y 的值:{y}\n")

alpha = torch.tensor([0.1,0.2,0.3])
y.backward(alpha)
print(f"向量y输出的梯度:{x.grad}\n")

y 的值:tensor([1., 4., 9.], grad_fn=<PowBackward0>)

向量y输出的梯度:tensor([0.2000, 0.8000, 1.8000])



In [38]:
# no_grad() detach()

# 1. torch.no_grad() - 临时禁用梯度计算
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

with torch.no_grad():
    y = x**2  # 这个操作不会被记录到计算图中,因为with torch.no_grad() 会禁用梯度计算
    print(f"在 no_grad() 内的 y 值:{y}\n")
    print(f"y.requires_grad 的值:{y.requires_grad}\n")

# 验证计算图没有记录
try:
    y.sum().backward()  # 会失败
except RuntimeError as e:
    print(f"错误信息:{e}")

y = x**2  # 这个操作不会计算梯度,因为with torch.no_grad() 会禁用梯度计算
print(f"在 no_grad() 内的 y 值:{y}\n")
print(f"y.requires_grad 的值:{y.requires_grad}\n")


# 2. detach() - 分离张量
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True).detach()
y = x**2
z = y.detach()  # 创建一个新的张量,与原计算图是分离的
print(f"原始张量y.requires_grad:{y.requires_grad}\n")
print(f"x分离后的 requires_grad 的值:{x.requires_grad}\n")


# 3. detach() 与 no_grad() 的区别
def compare_detach_and_nograd():
    x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

    # 使用 detach()
    y =  x ** 2
    z1 = y.detach()  # 创建一个新的张量,与原计算图是分离的
    

    # 使用 no_grad()
    with torch.no_grad():
        y =  x ** 2
        z2 = y.detach()  # z2 从一开始就不存在计算图中

    print(f"z1 的值:{z1}\n")
    print(f"z2 的值:{z2}\n")

compare_detach_and_nograd()


在 no_grad() 内的 y 值:tensor([1., 4., 9.])

y.requires_grad 的值:False

错误信息:element 0 of tensors does not require grad and does not have a grad_fn
在 no_grad() 内的 y 值:tensor([1., 4., 9.], grad_fn=<PowBackward0>)

y.requires_grad 的值:True

原始张量y.requires_grad:False

x分离后的 requires_grad 的值:False

z1 的值:tensor([1., 4., 9.])

z2 的值:tensor([1., 4., 9.])

